# Notebook 20 — Project Dataset Pipeline

Castalia Mentor starts with a data pipeline, not a trainer. In this capstone notebook you will turn raw tutoring examples into versioned artifacts that are ready for supervised finetuning, optional preference alignment, and later benchmark work.

## What you will build

- a first-principles schema for Castalia Mentor examples
- a small raw corpus with deliberate noise, duplicates, and edge cases
- cleaning, validation, and synthesis hooks that stay transparent
- train, validation, and test splits with leakage-aware grouping
- versioned JSONL, CSV, and Hugging Face dataset artifacts for later notebooks

## Design principle

The point is not to hide preprocessing behind a platform. The point is to make every transformation visible: what entered the dataset, what got removed, what got synthesized, how the split was made, and which exact artifact version will feed notebook 21.

In [ ]:
# --- Capstone dataset pipeline setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q datasets pandas matplotlib scikit-learn

import hashlib
import json
import math
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from datasets import Dataset, DatasetDict, load_from_disk
from sklearn.model_selection import GroupShuffleSplit

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)
random.seed(20)

print("Capstone dataset setup ready")

## Step 1: Add project helpers and artifact paths

We keep the helpers explicit so the data contract stays easy to audit and reuse.

In [ ]:
PROJECT_NAME = "Castalia Mentor"
ARTIFACT_DIR = Path("artifacts") / "notebook_20_project_dataset_pipeline"
VERSION_ROOT = ARTIFACT_DIR / "versions"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
VERSION_ROOT.mkdir(parents=True, exist_ok=True)

def normalize_text(text):
    text = text or ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\u00a0", " ")
    return " ".join(text.strip().split())

def stable_hash(value, length=12):
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()[:length]

def approx_tokens(text):
    words = normalize_text(text).split()
    return max(1, math.ceil(len(words) * 1.3))

def jaccard_similarity(left, right):
    left_tokens = set(normalize_text(left).lower().split())
    right_tokens = set(normalize_text(right).lower().split())
    if not left_tokens or not right_tokens:
        return 0.0
    return len(left_tokens & right_tokens) / len(left_tokens | right_tokens)

def first_sentence(text):
    text = normalize_text(text)
    parts = [part.strip() for part in text.split(".") if part.strip()]
    return parts[0] + "." if parts else text

def write_jsonl(path_obj, records):
    path_obj.parent.mkdir(parents=True, exist_ok=True)
    with open(path_obj, "w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Artifacts:", ARTIFACT_DIR.resolve())

## Step 2: Define the example schema and quality contract

The schema should be rich enough for later training and evaluation, but still compact enough to inspect by hand.

In [ ]:
schema_fields = [
    {"field": "example_id", "required": True, "description": "Stable content-derived identifier used across artifacts."},
    {"field": "source_track", "required": True, "description": "Castalia track where the seed example came from."},
    {"field": "scenario_family", "required": True, "description": "Leakage-aware grouping key for splits and later regression slices."},
    {"field": "mentor_intent", "required": True, "description": "Why the mentor response exists: explain, compare, troubleshoot, calibrate, and so on."},
    {"field": "difficulty", "required": True, "description": "Foundation, core, or stretch difficulty band."},
    {"field": "prompt", "required": True, "description": "User-side tutoring request after normalization."},
    {"field": "response", "required": True, "description": "Assistant-side mentor answer after normalization."},
    {"field": "rubric_tags", "required": True, "description": "Evaluation and training tags reused later for slicing."},
    {"field": "origin", "required": True, "description": "Seed or synthetic provenance marker."},
    {"field": "license", "required": True, "description": "Open provenance note for release hygiene."},
    {"field": "messages", "required": False, "description": "Chat-style record for SFT pipelines."},
    {"field": "text", "required": False, "description": "Rendered training string for notebook-native use."},
]

quality_contract = [
    {"rule": "non_empty_fields", "meaning": "Prompt and response must both survive normalization."},
    {"rule": "mentor_specificity", "meaning": "Responses should teach, compare, or guide rather than answer vaguely."},
    {"rule": "deduplicate", "meaning": "Exact and near-duplicate examples should not silently overweight one pattern."},
    {"rule": "grouped_split", "meaning": "Examples from one scenario family stay inside one split."},
    {"rule": "version_everything", "meaning": "Every export has a dataset version, manifest, and reproducible counts."},
]

schema_df = pd.DataFrame(schema_fields)
quality_df = pd.DataFrame(quality_contract)
display(schema_df)
display(quality_df)

## Step 3: Build a noisy raw project corpus

These examples imitate the kind of repository-native tutoring data a real project starts with: good seeds mixed with duplicates, whitespace drift, HTML fragments, and one empty response.

In [ ]:
raw_records = [
    {
        "source_id": "pe_01",
        "source_track": "prompt-engineering",
        "scenario_family": "planning_breakdown",
        "mentor_intent": "teach decomposition",
        "difficulty": "foundation",
        "prompt": "A student keeps writing one giant prompt. How should Castalia Mentor teach decomposition?",
        "response": "Start by separating goal, constraints, plan, and verification. Show one tiny example, then ask the student to rewrite the task in those four parts before they expand it.",
        "rubric_tags": ["prompting", "decomposition", "teaching"],
        "license": "castalia-original",
        "source_type": "handwritten",
    },
    {
        "source_id": "pe_01_dup",
        "source_track": "prompt-engineering",
        "scenario_family": "planning_breakdown",
        "mentor_intent": "teach decomposition",
        "difficulty": "foundation",
        "prompt": "A student keeps writing one giant prompt. How should Castalia Mentor teach decomposition?",
        "response": "Start by separating goal, constraints, plan, and verification. Show one tiny example, then ask the student to rewrite the task in those four parts before they expand it.",
        "rubric_tags": ["prompting", "decomposition", "teaching"],
        "license": "castalia-original",
        "source_type": "duplicate_exact",
    },
    {
        "source_id": "rag_01",
        "source_track": "rag",
        "scenario_family": "grounding_gap",
        "mentor_intent": "teach evidence use",
        "difficulty": "core",
        "prompt": "The retrieved notes disagree with each other. What should the mentor do before answering?",
        "response": "Say what the notes agree on, flag the contradiction, and ask for one more retrieval pass or a stronger source before claiming a final answer.",
        "rubric_tags": ["rag", "grounding", "uncertainty"],
        "license": "castalia-original",
        "source_type": "handwritten",
    },
    {
        "source_id": "rag_01_html",
        "source_track": "rag",
        "scenario_family": "grounding_gap",
        "mentor_intent": "teach evidence use",
        "difficulty": "core",
        "prompt": "The retrieved notes disagree with each other. What should the mentor do before answering?",
        "response": "<p>Say what the notes agree on, flag the contradiction, and ask for one more retrieval pass or a stronger source before claiming a final answer.</p>",
        "rubric_tags": ["RAG", "grounding", "uncertainty"],
        "license": "castalia-original",
        "source_type": "html_noise",
    },
    {
        "source_id": "agents_01",
        "source_track": "agents",
        "scenario_family": "tool_retry",
        "mentor_intent": "teach tool recovery",
        "difficulty": "stretch",
        "prompt": "An agent step timed out while calling a documentation tool. How should the mentor explain the recovery plan?",
        "response": "Name the failed step, keep the partial observations, retry with a narrower query, and only escalate after the second attempt still fails. Students should see the debugging loop, not just the final answer.",
        "rubric_tags": ["agents", "debugging", "tool-use"],
        "license": "castalia-original",
        "source_type": "handwritten",
    },
    {
        "source_id": "agents_01_empty",
        "source_track": "agents",
        "scenario_family": "tool_retry",
        "mentor_intent": "teach tool recovery",
        "difficulty": "stretch",
        "prompt": "An agent step timed out while calling a documentation tool. How should the mentor explain the recovery plan?",
        "response": "   ",
        "rubric_tags": ["agents", "debugging", "tool-use"],
        "license": "castalia-original",
        "source_type": "empty_response",
    },
    {
        "source_id": "evals_01",
        "source_track": "evals",
        "scenario_family": "rubric_design",
        "mentor_intent": "teach scoring",
        "difficulty": "core",
        "prompt": "How would you teach a student to design a tiny rubric for tutoring answers?",
        "response": "Pick two or three dimensions that matter, define what a pass looks like, and score the same prompt the same way for both base and tuned models so regressions stay visible.",
        "rubric_tags": ["evals", "rubrics", "comparison"],
        "license": "castalia-original",
        "source_type": "handwritten",
    },
    {
        "source_id": "evals_01_near",
        "source_track": "evals",
        "scenario_family": "rubric_design",
        "mentor_intent": "teach scoring",
        "difficulty": "core",
        "prompt": "How would you teach a student to design a small rubric for tutoring answers?",
        "response": "Pick two or three dimensions that matter, define what passing looks like, and use the same prompts for the base model and the tuned model so regressions remain visible.",
        "rubric_tags": ["evals", "rubrics", "comparison"],
        "license": "castalia-original",
        "source_type": "duplicate_near",
    },
    {
        "source_id": "ft_01",
        "source_track": "finetuning",
        "scenario_family": "qlora_recipe",
        "mentor_intent": "teach first run scoping",
        "difficulty": "foundation",
        "prompt": "What is a safe first QLoRA recipe on a T4 for Castalia Mentor?",
        "response": "Use a small open instruct model, 4-bit loading, a short held-out split, and a deliberately short training run before you spend time on bigger ablations.",
        "rubric_tags": ["finetuning", "qlora", "colab"],
        "license": "castalia-original",
        "source_type": "handwritten",
    },
    {
        "source_id": "ft_01_dup",
        "source_track": "finetuning",
        "scenario_family": "qlora_recipe",
        "mentor_intent": "teach first run scoping",
        "difficulty": "foundation",
        "prompt": "  What is a safe first QLoRA recipe on a T4 for Castalia Mentor?  ",
        "response": "Use a small open instruct model, 4-bit loading, a short held-out split, and a deliberately short training run before you spend time on bigger ablations. ",
        "rubric_tags": ["finetuning", "qlora", "colab"],
        "license": "castalia-original",
        "source_type": "duplicate_exact",
    },
    {
        "source_id": "ft_02",
        "source_track": "finetuning",
        "scenario_family": "dataset_card",
        "mentor_intent": "teach artifact hygiene",
        "difficulty": "core",
        "prompt": "What belongs in a dataset card for a mentor model project?",
        "response": "Document where the data came from, what got filtered out, the intended use, the known blind spots, and which evaluation slices should block release if they regress.",
        "rubric_tags": ["finetuning", "artifacts", "governance"],
        "license": "castalia-original",
        "source_type": "handwritten",
    },
    {
        "source_id": "ft_02_html",
        "source_track": "finetuning",
        "scenario_family": "dataset_card",
        "mentor_intent": "teach artifact hygiene",
        "difficulty": "core",
        "prompt": "What belongs in a dataset card for a mentor model project?",
        "response": "<div>Document where the data came from, what got filtered out, the intended use, the known blind spots, and which evaluation slices should block release if they regress.</div>",
        "rubric_tags": ["finetuning", "artifacts", "governance"],
        "license": "castalia-original",
        "source_type": "html_noise",
    },
    {
        "source_id": "ft_03",
        "source_track": "finetuning",
        "scenario_family": "preference_pairs",
        "mentor_intent": "prepare alignment",
        "difficulty": "stretch",
        "prompt": "How should Castalia Mentor turn human review comments into preference pairs?",
        "response": "Keep the same prompt, write down the stronger and weaker answers, and annotate what the stronger one did better so later failure analysis has a reason instead of only a winner.",
        "rubric_tags": ["alignment", "preferences", "annotation"],
        "license": "castalia-original",
        "source_type": "handwritten",
    },
    {
        "source_id": "evals_02",
        "source_track": "evals",
        "scenario_family": "uncertainty_boundary",
        "mentor_intent": "teach calibration",
        "difficulty": "core",
        "prompt": "What should the mentor say when it does not know a detail with confidence?",
        "response": "State the uncertainty plainly, share the partial reasoning that is still reliable, and suggest the next source or experiment that would resolve the gap.",
        "rubric_tags": ["evals", "calibration", "safety"],
        "license": "castalia-original",
        "source_type": "handwritten",
    },
    {
        "source_id": "ft_04",
        "source_track": "finetuning",
        "scenario_family": "export_handoff",
        "mentor_intent": "teach deployment trade-offs",
        "difficulty": "core",
        "prompt": "How should a student decide between keeping an adapter separate and merging it?",
        "response": "Keep the adapter separate when rollback, multi-task reuse, or small artifact size matters. Merge when the downstream deployment path wants one checkpoint and the benchmark evidence is already strong.",
        "rubric_tags": ["deployment", "adapters", "benchmarking"],
        "license": "castalia-original",
        "source_type": "handwritten",
    },
    {
        "source_id": "strategy_01",
        "source_track": "prompt-engineering",
        "scenario_family": "rag_vs_finetune",
        "mentor_intent": "teach strategy choice",
        "difficulty": "foundation",
        "prompt": "When should a student choose RAG over finetuning for Castalia Mentor?",
        "response": "Use RAG when the main issue is missing or changing knowledge. Use finetuning when the repeated problem is behavior, format, pedagogy, or stable task style that retrieval alone does not fix.",
        "rubric_tags": ["strategy", "rag", "finetuning"],
        "license": "castalia-original",
        "source_type": "handwritten",
    },
    {
        "source_id": "ft_05",
        "source_track": "finetuning",
        "scenario_family": "synthetic_review",
        "mentor_intent": "teach synthesis discipline",
        "difficulty": "stretch",
        "prompt": "What warning should the mentor give before a team scales up synthetic data?",
        "response": "Synthetic data helps only after filtering, deduplication, and held-out evaluation are already in place. More generated text is not progress if no one checks what it teaches the model.",
        "rubric_tags": ["synthetic-data", "filtering", "evaluation"],
        "license": "castalia-original",
        "source_type": "handwritten",
    },
    {
        "source_id": "agents_02",
        "source_track": "agents",
        "scenario_family": "tool_trace",
        "mentor_intent": "teach observability",
        "difficulty": "foundation",
        "prompt": "Why should the mentor keep a trace of tool calls during an experiment?",
        "response": "A trace shows which step used which tool, what evidence came back, and where the failure started. That makes debugging teachable instead of mysterious.",
        "rubric_tags": ["agents", "observability", "debugging"],
        "license": "castalia-original",
        "source_type": "handwritten",
    },
]

raw_df = pd.DataFrame(raw_records)
display(raw_df[["source_id", "source_track", "scenario_family", "source_type"]].head(12))
print("Raw rows:", len(raw_df))

## Step 4: Profile the raw corpus before cleaning

Measurement comes first. A good pipeline makes the starting mess visible before it claims improvement.

In [ ]:
raw_df["prompt_tokens"] = raw_df["prompt"].map(approx_tokens)
raw_df["response_tokens"] = raw_df["response"].map(approx_tokens)

raw_profile = pd.DataFrame(
    [
        {"metric": "raw_rows", "value": int(len(raw_df))},
        {"metric": "empty_responses", "value": int((raw_df["response"].fillna("").str.strip() == "").sum())},
        {"metric": "exact_prompt_response_duplicates", "value": int(raw_df.duplicated(subset=["prompt", "response"]).sum())},
        {"metric": "tracks", "value": int(raw_df["source_track"].nunique())},
        {"metric": "scenario_families", "value": int(raw_df["scenario_family"].nunique())},
        {"metric": "mean_prompt_tokens", "value": round(raw_df["prompt_tokens"].mean(), 1)},
        {"metric": "mean_response_tokens", "value": round(raw_df["response_tokens"].mean(), 1)},
    ]
)

display(raw_profile)
display(raw_df["source_track"].value_counts().rename_axis("source_track").reset_index(name="rows"))

## Step 5: Canonicalize the records and validate required fields

This is where loose source records become training-ready examples with stable identifiers and reusable metadata.

In [ ]:
required_fields = [row["field"] for row in schema_fields if row["required"]]

canonical_rows = []
validation_rows = []
for row in raw_records:
    canonical = {
        "source_id": row["source_id"],
        "source_track": row["source_track"].strip().lower(),
        "scenario_family": row["scenario_family"].strip().lower(),
        "mentor_intent": row["mentor_intent"].strip().lower(),
        "difficulty": row["difficulty"].strip().lower(),
        "prompt": normalize_text(row["prompt"]),
        "response": normalize_text(row["response"]),
        "rubric_tags": sorted({tag.strip().lower() for tag in row["rubric_tags"]}),
        "license": row.get("license", "castalia-original"),
        "origin": "seed",
        "source_type": row.get("source_type", "handwritten"),
    }
    canonical["example_id"] = stable_hash(
        canonical["source_track"] + "||" + canonical["scenario_family"] + "||" + canonical["prompt"] + "||" + canonical["response"],
        length=16,
    )
    canonical["fingerprint"] = stable_hash(canonical["prompt"] + "||" + canonical["response"], length=20)
    canonical["approx_prompt_tokens"] = approx_tokens(canonical["prompt"])
    canonical["approx_response_tokens"] = approx_tokens(canonical["response"])
    canonical_rows.append(canonical)

    validation_rows.append(
        {
            "source_id": canonical["source_id"],
            "has_all_required_fields": all(bool(canonical.get(field)) for field in required_fields if field != "origin"),
            "response_long_enough": len(canonical["response"].split()) >= 12,
            "teaching_signal_present": any(
                phrase in canonical["response"].lower()
                for phrase in ["show", "ask", "use", "keep", "state", "document", "compare", "retry", "split"]
            ),
        }
    )

canonical_df = pd.DataFrame(canonical_rows)
validation_df = pd.DataFrame(validation_rows)
display(canonical_df.head(8))
display(validation_df.head(8))

## Step 6: Remove structurally invalid rows

We drop rows that fail the most basic contract before any clever filtering happens.

In [ ]:
structural_mask = (
    (canonical_df["prompt"].str.len() > 20)
    & (canonical_df["response"].str.len() > 40)
)

structural_df = canonical_df[structural_mask].copy()
removed_structural_df = canonical_df[~structural_mask].copy()

display(pd.DataFrame(
    [
        {"stage": "canonical", "rows": len(canonical_df)},
        {"stage": "structural_filter", "rows": len(structural_df)},
    ]
))
display(removed_structural_df[["source_id", "source_track", "prompt", "response"]])

## Step 7: Remove exact and near duplicates

Small tutoring datasets are highly sensitive to repetition, so we do both a strict fingerprint pass and a softer lexical near-duplicate pass.

In [ ]:
dedup_exact_df = structural_df.drop_duplicates(subset=["fingerprint"]).copy()
exact_removed = len(structural_df) - len(dedup_exact_df)

kept_rows = []
near_duplicate_rows = []
for row in dedup_exact_df.sort_values("source_id").to_dict("records"):
    duplicate_match = None
    for kept in kept_rows:
        same_family = row["scenario_family"] == kept["scenario_family"]
        prompt_sim = jaccard_similarity(row["prompt"], kept["prompt"])
        response_sim = jaccard_similarity(row["response"], kept["response"])
        if same_family and prompt_sim >= 0.80 and response_sim >= 0.72:
            duplicate_match = {
                "drop_source_id": row["source_id"],
                "keep_source_id": kept["source_id"],
                "prompt_similarity": round(prompt_sim, 3),
                "response_similarity": round(response_sim, 3),
            }
            break
    if duplicate_match:
        near_duplicate_rows.append(duplicate_match)
    else:
        kept_rows.append(row)

seed_df = pd.DataFrame(kept_rows).reset_index(drop=True)
near_dup_df = pd.DataFrame(near_duplicate_rows)

display(pd.DataFrame([{"exact_duplicates_removed": int(exact_removed), "near_duplicates_removed": int(len(near_duplicate_rows)), "seed_rows_kept": int(len(seed_df))}]))
display(near_dup_df if not near_dup_df.empty else pd.DataFrame({"message": ["No near duplicates found"]}))

## Step 8: Define synthesis hooks for controlled augmentation

These hooks are deterministic on purpose. They are not trying to replace a teacher model. They are creating traceable augmentation patterns that notebook 21 can later expand or replace.

In [ ]:
synthesis_hooks = [
    {
        "hook_name": "checklist_reframe",
        "prompt_prefix": "Turn this tutoring advice into a short checklist for a student.",
        "response_builder": lambda row: "Checklist: 1) " + first_sentence(row["response"]) + " 2) Name the key evidence or constraint. 3) End with the next experiment or review step.",
    },
    {
        "hook_name": "socratic_hint",
        "prompt_prefix": "Rewrite this as a hint-first mentoring exchange instead of a direct answer.",
        "response_builder": lambda row: "Start with one diagnostic question. Then offer a hint based on " + first_sentence(row["response"]).lower() + " End by asking the student to try the next step themselves.",
    },
    {
        "hook_name": "misconception_repair",
        "prompt_prefix": "A student has the wrong instinct here. Correct it gently and explain the better move.",
        "response_builder": lambda row: "Common mistake: jumping straight to the conclusion. Better move: " + row["response"] + " Finish by naming the specific misconception that changed.",
    },
]

display(pd.DataFrame([{key: value for key, value in hook.items() if key != "response_builder"} for hook in synthesis_hooks]))

## Step 9: Generate candidate synthetic examples and filter them

The goal is not maximum volume. The goal is a few high-signal variants per seed example with clear provenance.

In [ ]:
synthetic_candidates = []
for row in seed_df.to_dict("records"):
    active_hooks = synthesis_hooks[:2] if row["difficulty"] == "foundation" else synthesis_hooks
    for hook in active_hooks:
        prompt = hook["prompt_prefix"] + " Original student request: " + row["prompt"]
        response = hook["response_builder"](row)
        synthetic_candidates.append(
            {
                "source_id": row["source_id"] + "__" + hook["hook_name"],
                "source_track": row["source_track"],
                "scenario_family": row["scenario_family"],
                "mentor_intent": row["mentor_intent"],
                "difficulty": row["difficulty"],
                "prompt": normalize_text(prompt),
                "response": normalize_text(response),
                "rubric_tags": sorted(set(row["rubric_tags"] + [hook["hook_name"], "synthetic"])),
                "license": row["license"],
                "origin": "synthetic",
                "source_type": "template_" + hook["hook_name"],
            }
        )

filtered_synthetic = []
synthetic_filter_rows = []
for row in synthetic_candidates:
    similarity_to_seed = max(
        jaccard_similarity(row["response"], seed_row["response"])
        for seed_row in seed_df[seed_df["scenario_family"] == row["scenario_family"]].to_dict("records")
    )
    keep = (
        len(row["response"].split()) >= 18
        and len(row["prompt"].split()) >= 10
        and similarity_to_seed < 0.94
    )
    row["example_id"] = stable_hash(row["source_track"] + "||" + row["scenario_family"] + "||" + row["prompt"] + "||" + row["response"], length=16)
    row["fingerprint"] = stable_hash(row["prompt"] + "||" + row["response"], length=20)
    row["approx_prompt_tokens"] = approx_tokens(row["prompt"])
    row["approx_response_tokens"] = approx_tokens(row["response"])
    synthetic_filter_rows.append(
        {
            "source_id": row["source_id"],
            "scenario_family": row["scenario_family"],
            "response_similarity_to_seed": round(similarity_to_seed, 3),
            "keep": keep,
        }
    )
    if keep:
        filtered_synthetic.append(row)

synthetic_df = pd.DataFrame(filtered_synthetic)
synthetic_filter_df = pd.DataFrame(synthetic_filter_rows)

display(synthetic_filter_df.head(12))
print("Synthetic rows kept:", len(synthetic_df))

## Step 10: Merge seed and synthetic data into one project table

At this stage the dataset becomes a versionable artifact rather than a loose collection of examples.

In [ ]:
project_df = pd.concat([seed_df, synthetic_df], ignore_index=True).sort_values(
    ["source_track", "scenario_family", "origin", "source_id"]
).reset_index(drop=True)

version_seed = "|".join(sorted(project_df["example_id"].tolist()))
DATASET_VERSION = "castalia_mentor_" + stable_hash(version_seed, length=10)
project_df["dataset_version"] = DATASET_VERSION

manifest = {
    "project": PROJECT_NAME,
    "dataset_version": DATASET_VERSION,
    "seed_rows": int(len(seed_df)),
    "synthetic_rows": int(len(synthetic_df)),
    "total_rows": int(len(project_df)),
    "tracks": sorted(project_df["source_track"].unique().tolist()),
    "scenario_families": sorted(project_df["scenario_family"].unique().tolist()),
}

display(pd.DataFrame([manifest]))
display(project_df[["source_id", "source_track", "scenario_family", "origin"]].head(12))

## Step 11: Create train, validation, and test splits with grouped leakage control

Scenario families stay intact across splits so the model does not see one phrasing of a task during training and a trivial rewording of the same task during evaluation.

In [ ]:
outer_splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=20)
train_val_indices, test_indices = next(outer_splitter.split(project_df, groups=project_df["scenario_family"]))

train_val_df = project_df.iloc[train_val_indices].reset_index(drop=True)
test_df = project_df.iloc[test_indices].reset_index(drop=True)

inner_splitter = GroupShuffleSplit(n_splits=1, test_size=0.19, random_state=21)
train_indices, validation_indices = next(inner_splitter.split(train_val_df, groups=train_val_df["scenario_family"]))

train_df = train_val_df.iloc[train_indices].reset_index(drop=True)
validation_df = train_val_df.iloc[validation_indices].reset_index(drop=True)

train_df["split"] = "train"
validation_df["split"] = "validation"
test_df["split"] = "test"

split_df = pd.concat([train_df, validation_df, test_df], ignore_index=True)

display(pd.DataFrame(
    [
        {"split": "train", "rows": len(train_df), "scenario_families": train_df["scenario_family"].nunique()},
        {"split": "validation", "rows": len(validation_df), "scenario_families": validation_df["scenario_family"].nunique()},
        {"split": "test", "rows": len(test_df), "scenario_families": test_df["scenario_family"].nunique()},
    ]
))

## Step 12: Inspect split balance and check for obvious leakage

Counts alone are not enough. We also want a quick lexical check that prompts across splits are not nearly identical.

In [ ]:
split_balance_df = (
    split_df.groupby(["split", "source_track"])
    .size()
    .reset_index(name="rows")
    .sort_values(["split", "rows"], ascending=[True, False])
)
display(split_balance_df)

cross_split_rows = []
for left_split in ["train", "validation", "test"]:
    left_rows = split_df[split_df["split"] == left_split].to_dict("records")
    other_rows = split_df[split_df["split"] != left_split].to_dict("records")
    max_prompt_overlap = 0.0
    for left_row in left_rows:
        for right_row in other_rows:
            overlap = jaccard_similarity(left_row["prompt"], right_row["prompt"])
            max_prompt_overlap = max(max_prompt_overlap, overlap)
    cross_split_rows.append({"split": left_split, "max_prompt_overlap_to_other_splits": round(max_prompt_overlap, 3)})

leakage_check_df = pd.DataFrame(cross_split_rows)
display(leakage_check_df)

ax = split_balance_df.pivot(index="source_track", columns="split", values="rows").fillna(0).plot(
    kind="bar",
    figsize=(9, 4),
    title="Rows by split and source track",
)
ax.set_ylabel("rows")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## Step 13: Render SFT records and optional preference-alignment hooks

Notebook 21 will want both ordinary supervised examples and optional preference pairs. We build both now so later experiments can stay reproducible.

In [ ]:
SYSTEM_PROMPT = "You are Castalia Mentor, a domain-specialized mentor. Teach clearly, stay evidence-aware, and state uncertainty honestly."

def render_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": row["response"]},
    ]

def render_text(messages):
    parts = []
    for message in messages:
        parts.append("<|" + message["role"] + "|>\n" + message["content"])
    return "\n\n".join(parts)

def degrade_response(text):
    sentence = first_sentence(text)
    degraded = sentence.replace("and", ",").replace("should", "can")
    return "Quick answer: " + degraded.replace("State", "Skip").replace("state", "skip")

def to_training_record(row):
    messages = render_messages(row)
    return {
        "example_id": row["example_id"],
        "dataset_version": row["dataset_version"],
        "source_track": row["source_track"],
        "scenario_family": row["scenario_family"],
        "mentor_intent": row["mentor_intent"],
        "difficulty": row["difficulty"],
        "origin": row["origin"],
        "rubric_tags": row["rubric_tags"],
        "messages": messages,
        "text": render_text(messages),
    }

train_records = [to_training_record(row) for row in train_df.to_dict("records")]
validation_records = [to_training_record(row) for row in validation_df.to_dict("records")]
test_records = [to_training_record(row) for row in test_df.to_dict("records")]

preference_records = []
for row in train_df.to_dict("records"):
    preference_records.append(
        {
            "example_id": row["example_id"],
            "dataset_version": row["dataset_version"],
            "prompt": row["prompt"],
            "chosen": row["response"],
            "rejected": degrade_response(row["response"]),
            "scenario_family": row["scenario_family"],
            "rubric_tags": row["rubric_tags"],
        }
    )

sft_dataset = DatasetDict(
    {
        "train": Dataset.from_list(train_records),
        "validation": Dataset.from_list(validation_records),
        "test": Dataset.from_list(test_records),
    }
)

display(pd.DataFrame(train_records)[["example_id", "source_track", "scenario_family", "origin"]].head())
display(pd.DataFrame(preference_records)[["example_id", "scenario_family", "prompt"]].head())

## Step 14: Export versioned JSONL, CSV, and dataset artifacts

This is the release point for the data pipeline. Everything downstream should refer to the version string rather than a loose notebook state.

In [ ]:
version_dir = VERSION_ROOT / DATASET_VERSION
split_dir = version_dir / "splits"
table_dir = version_dir / "tables"
hf_dir = version_dir / "hf_dataset"
version_dir.mkdir(parents=True, exist_ok=True)
split_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

for split_name, records in {
    "train": train_records,
    "validation": validation_records,
    "test": test_records,
}.items():
    write_jsonl(split_dir / (split_name + ".jsonl"), records)
    pd.DataFrame(records).to_csv(table_dir / (split_name + ".csv"), index=False)

write_jsonl(version_dir / "preference_pairs.jsonl", preference_records)
pd.DataFrame(project_df).to_csv(table_dir / "full_project_table.csv", index=False)

with open(version_dir / "manifest.json", "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)
with open(version_dir / "schema.json", "w", encoding="utf-8") as handle:
    json.dump(schema_fields, handle, indent=2)
with open(version_dir / "quality_contract.json", "w", encoding="utf-8") as handle:
    json.dump(quality_contract, handle, indent=2)

sft_dataset.save_to_disk(str(hf_dir))

print("Version directory:", version_dir.resolve())
print("Saved dataset version:", DATASET_VERSION)

## Step 15: Reload the artifacts and validate the export contract

Never trust an export just because the write step succeeded. Reload it and compare counts immediately.

In [ ]:
reloaded_manifest = json.loads((version_dir / "manifest.json").read_text(encoding="utf-8"))
reloaded_train_jsonl = [json.loads(line) for line in (split_dir / "train.jsonl").read_text(encoding="utf-8").splitlines()]
reloaded_validation_jsonl = [json.loads(line) for line in (split_dir / "validation.jsonl").read_text(encoding="utf-8").splitlines()]
reloaded_test_jsonl = [json.loads(line) for line in (split_dir / "test.jsonl").read_text(encoding="utf-8").splitlines()]
reloaded_preferences = [json.loads(line) for line in (version_dir / "preference_pairs.jsonl").read_text(encoding="utf-8").splitlines()]
reloaded_dataset = load_from_disk(str(hf_dir))

validation_checks = pd.DataFrame(
    [
        {"check": "manifest_total_rows", "value": reloaded_manifest["total_rows"], "expected": len(project_df)},
        {"check": "train_jsonl_rows", "value": len(reloaded_train_jsonl), "expected": len(train_records)},
        {"check": "validation_jsonl_rows", "value": len(reloaded_validation_jsonl), "expected": len(validation_records)},
        {"check": "test_jsonl_rows", "value": len(reloaded_test_jsonl), "expected": len(test_records)},
        {"check": "preference_rows", "value": len(reloaded_preferences), "expected": len(preference_records)},
        {"check": "hf_train_rows", "value": len(reloaded_dataset["train"]), "expected": len(train_records)},
    ]
)
validation_checks["pass"] = validation_checks["value"] == validation_checks["expected"]
display(validation_checks)

## Step 16: Inspect one end-to-end training record

This final spot check makes the downstream contract tangible before notebook 21 starts scheduling runs.

In [ ]:
sample_training_record = train_records[0]
sample_preference_record = preference_records[0]

print("Training record")
print(json.dumps(sample_training_record, indent=2, ensure_ascii=False))
print()
print("Preference record")
print(json.dumps(sample_preference_record, indent=2, ensure_ascii=False))

## Recap

You now have a reproducible Castalia Mentor dataset pipeline with:

- a visible schema and quality contract
- cleaned seed examples plus controlled synthesis hooks
- grouped train, validation, and test splits
- optional preference pairs for later alignment work
- versioned artifacts that notebook 21 can load directly